In [1]:
import os
import glob
import pandas as pd
import numpy as np
from astropy.io import fits

In [16]:
FITS_DIR = "./downloaded-koi-lcss/"

In [17]:
def extract_lc_arrays(fits_filepath):
    """Extracts time, flux, flux_err, and n_points from a Kepler .fits file."""
    try:
        with fits.open(fits_filepath) as hdul:
            # The lightcurve data is typically in the first extension
            data = hdul[1].data
            
            # Filter out NaNs if necessary
            mask = ~np.isnan(data['TIME']) & ~np.isnan(data['PDCSAP_FLUX'])
            
            time = data['TIME'][mask]
            flux = data['PDCSAP_FLUX'][mask]
            flux_err = data['PDCSAP_FLUX_ERR'][mask]
            n_points = len(time)
            
            # Extract KIC from the header or filename if needed
            kic = hdul[0].header.get('KEPLERID', 'Unknown')
            
            return pd.Series({
                "KIC": kic,
                "time": time,
                "flux": flux,
                "flux_err": flux_err,
                "n_points": n_points
            })
    except Exception as e:
        print(f"Error processing {fits_filepath}: {e}")
        return pd.Series({
            "KIC": np.nan,
            "time": np.array([]), 
            "flux": np.array([]), 
            "flux_err": np.array([]), 
            "n_points": 0
        })

In [18]:
fits_files = glob.glob(os.path.join(FITS_DIR, "*_llc.fits"))

In [19]:
print(f"Extracting data from {len(fits_files)} files...")
df_koi = pd.DataFrame({'filepath': fits_files})

Extracting data from 32 files...


In [20]:
extracted_features = df_koi['filepath'].apply(extract_lc_arrays)

In [21]:
df_final = extracted_features.dropna(subset=['KIC']).set_index('KIC')

In [23]:
output_pickle = "koi_cumulative_lcs.pkl"
df_final.to_pickle(output_pickle)